Write a notebook setup cell that:
- imports sys, pathlib.Path, and pandas;
- prints the active Python executable;
- finds the project root by searching the current working directory
  and its parents for pyproject.toml;
- defines PROJECT_ROOT and RAW_DIR = PROJECT_ROOT / "data" / "raw";
- raises a clear error if the root or raw directory cannot be found;
- displays the CSV filenames and their sizes in MB in a sorted DataFrame.

Do not use __file__, change the working directory, install packages,
or modify any data files.

In [1]:
import sys
from pathlib import Path
import pandas as pd

print(f"Active Python executable: {sys.executable}")

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find pyproject.toml in {Path.cwd()} or any of its parent directories."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DIR}")

csv_files = sorted(RAW_DIR.glob("*.csv"), key=lambda path: path.name.lower())
csv_inventory = pd.DataFrame(
    [
        {"filename": path.name, "size_mb": path.stat().st_size / (1024**2)}
        for path in csv_files
    ],
    columns=["filename", "size_mb"],
)

display(csv_inventory)

Active Python executable: d:\Projects\projects\retail_forecasting\.venv\Scripts\python.exe


,filename,size_mb
0,holidays_events.csv,0.021276
1,oil.csv,0.019627
2,sample_submission.csv,0.326303
3,stores.csv,0.001323
4,test.csv,0.974912
5,train.csv,116.157887
6,transactions.csv,1.480710


Using the existing RAW_DIR, load stores.csv into a DataFrame named stores.

Display:
1. Its shape and first 10 rows.
2. Each column's dtype, missing-value count, and distinct-value count.
3. The number of duplicate store_nbr values.
4. The number of stores in each state, sorted descending.
5. A table grouped by state and city showing the number of unique stores.
6. A cross-tabulation of state versus cluster, counting unique stores.

Keep the original columns unchanged. Do not select a sample, fill missing
values, or remove rows. Use readable DataFrame outputs rather than
printing large tables as strings.

In [2]:
stores = pd.read_csv(RAW_DIR / "stores.csv")

print(f"Shape: {stores.shape}")
display(stores.head(10))

column_summary = pd.DataFrame({
    "dtype": stores.dtypes.astype(str),
    "missing_values": stores.isna().sum(),
    "distinct_values": stores.nunique(),
})
display(column_summary)

duplicate_store_nbr_count = stores["store_nbr"].duplicated().sum()
print(f"Duplicate store_nbr values: {duplicate_store_nbr_count}")

stores_per_state = (
    stores.groupby("state")["store_nbr"]
    .nunique()
    .sort_values(ascending=False)
    .rename("unique_stores")
    .to_frame()
)
display(stores_per_state)

stores_per_state_city = (
    stores.groupby(["state", "city"])["store_nbr"]
    .nunique()
    .rename("unique_stores")
    .reset_index()
)
display(stores_per_state_city)

state_cluster_crosstab = pd.crosstab(
    index=stores["state"],
    columns=stores["cluster"],
    values=stores["store_nbr"],
    aggfunc="nunique",
)
display(state_cluster_crosstab)

Shape: (54, 5)


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4
5,6,Quito,Pichincha,D,13
6,7,Quito,Pichincha,D,8
7,8,Quito,Pichincha,D,8
8,9,Quito,Pichincha,B,6
9,10,Quito,Pichincha,C,15


,dtype,missing_values,distinct_values
store_nbr,int64,0,54
city,str,0,22
state,str,0,16
type,str,0,5
cluster,int64,0,17


Duplicate store_nbr values: 0


,unique_stores
state,
Pichincha,19
Guayas,11
Manabi,3
Azuay,3
Santo Domingo de los Tsachilas,3
Los Rios,2
El Oro,2
Cotopaxi,2
Tungurahua,2


,state,city,unique_stores
0,Azuay,Cuenca,3
1,Bolivar,Guaranda,1
2,Chimborazo,Riobamba,1
3,Cotopaxi,Latacunga,2
4,El Oro,Machala,2
5,Esmeraldas,Esmeraldas,1
6,Guayas,Daule,1
7,Guayas,Guayaquil,8
8,Guayas,Libertad,1
9,Guayas,Playas,1


cluster,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
state,,,,,,,,,,,,,,,,,
Azuay,NaN,2.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Bolivar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
Chimborazo,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Cotopaxi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
El Oro,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Esmeraldas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Guayas,2.0,NaN,3.0,NaN,NaN,1.0,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0
Imbabura,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
Loja,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
